## GSAT trend patterns

In [ ]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [ ]:
import src.slurm_cluster as scluster
client, scluster = scluster.init_dask_slurm_cluster()

In [ ]:
# Input the model simulated trend
dir_unforced_input = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/{model}'

# store datasets in a dict and also create named variables for later cells
unforced_trend_std = {}

for model in ['CanESM5', 'CESM2', 'IPSL_CM6A', 'EC_Earth3', 'ACCESS', 'MPI_ESM', 'MIROC6']:
    unforced_trend_std_da = xr.open_dataset(dir_unforced_input.format(model=model) + f'/{model}_ICV_noise_std_trend_pattern_1850_2022.nc')
    unforced_trend_std[model] = unforced_trend_std_da
    globals()[f"{model}_unforced_trend_std_da"] = unforced_trend_std_da

In [ ]:
unforced_trend_std

In [ ]:
unforced_trend_std["ACCESS"]

In [ ]:
# calculate the single model ensemble mean first
CanESM5_unforced_trend_std_mean   = unforced_trend_std["CanESM5"].mean(dim='run')
CESM2_unforced_trend_std_mean     = unforced_trend_std["CESM2"].mean(dim='run')
IPSL_unforced_trend_std_mean      = unforced_trend_std["IPSL_CM6A"].mean(dim='run')
EC_Earth_unforced_trend_std_mean  = unforced_trend_std["EC_Earth3"].mean(dim='run')
ACCESS_unforced_trend_std_mean    = unforced_trend_std["ACCESS"].mean(dim='run')
MPI_ESM_unforced_trend_std_mean   = unforced_trend_std["MPI_ESM"].mean(dim='run')
MIROC6_unforced_trend_std_mean    = unforced_trend_std["MIROC6"].mean(dim='run')

In [ ]:
CanESM5_unforced_trend_std_mean

In [ ]:
CESM2_unforced_trend_std_mean

In [ ]:
# calculate the mean of SMILE std pattern
MMEM_unforced_trend_std_da = (CanESM5_unforced_trend_std_mean +
                                CESM2_unforced_trend_std_mean +
                              IPSL_unforced_trend_std_mean +
                              EC_Earth_unforced_trend_std_mean +
                              ACCESS_unforced_trend_std_mean +
                              MPI_ESM_unforced_trend_std_mean +
                              MIROC6_unforced_trend_std_mean) / 7.0

In [ ]:
MMEM_unforced_trend_std_da

In [ ]:
dir_out = "/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MMLE/"
MMEM_unforced_trend_std_da.to_netcdf(dir_out + 'MMLE_ICV_noise_std_trend_pattern_1850_2022.nc')

In [ ]:
# # save the data into netcdf file
# dir_out = '/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Supp_Figure2/data/'
# for interval, data in MMEM_unforced_trend_std_da.items():
#     data.to_netcdf(dir_out + 'MMEM_annual_' + interval + '_noise_trend_std.nc')

### Aggregate all the simulations of the models

In [ ]:
# read in the single realization trend data
# Input the observational trend
variable_name = ['10yr', '30yr', '60yr']

# Input the single model realization raw trend
dir_forced_input = '/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Supp_Figure2/data/'

CanESM5_forced_trend_da  = {}
IPSL_forced_trend_da     = {}
EC_Earth_forced_trend_da = {}
ACCESS_forced_trend_da   = {}
MPI_ESM_forced_trend_da  = {}
MIROC6_forced_trend_da   = {}

for interval in variable_name:
    CanESM5_forced_trend_da[interval]   = xr.open_mfdataset(dir_forced_input + 'CanESM5_annual_' + interval + '_run*_trend.nc',combine='nested', concat_dim='run')
    IPSL_forced_trend_da[interval]      = xr.open_mfdataset(dir_forced_input + 'IPSL_annual_' + interval + '_run*_trend.nc',combine='nested', concat_dim='run')
    EC_Earth_forced_trend_da[interval]  = xr.open_mfdataset(dir_forced_input + 'EC_Earth_annual_' + interval + '_run*_trend.nc',combine='nested', concat_dim='run')
    ACCESS_forced_trend_da[interval]    = xr.open_mfdataset(dir_forced_input + 'ACCESS_annual_' + interval + '_run*_trend.nc',combine='nested', concat_dim='run')
    MPI_ESM_forced_trend_da[interval]   = xr.open_mfdataset(dir_forced_input + 'MPI_ESM_annual_' + interval + '_run*_trend.nc',combine='nested', concat_dim='run')
    MIROC6_forced_trend_da[interval]    = xr.open_mfdataset(dir_forced_input + 'MIROC6_annual_' + interval + '_run*_trend.nc',combine='nested', concat_dim='run')

In [ ]:
CanESM5_forced_trend_da

In [ ]:
# rename the variable name
for interval in variable_name:
    CanESM5_forced_trend_da[interval] = CanESM5_forced_trend_da[interval].rename_vars({'__xarray_dataarray_variable__': 'trend'})
    IPSL_forced_trend_da[interval] = IPSL_forced_trend_da[interval].rename_vars({'__xarray_dataarray_variable__': 'trend'})
    EC_Earth_forced_trend_da[interval] = EC_Earth_forced_trend_da[interval].rename_vars({'__xarray_dataarray_variable__': 'trend'})
    ACCESS_forced_trend_da[interval] = ACCESS_forced_trend_da[interval].rename_vars({'__xarray_dataarray_variable__': 'trend'})
    MPI_ESM_forced_trend_da[interval] = MPI_ESM_forced_trend_da[interval].rename_vars({'__xarray_dataarray_variable__': 'trend'})
    MIROC6_forced_trend_da[interval] = MIROC6_forced_trend_da[interval].rename_vars({'__xarray_dataarray_variable__': 'trend'})

In [ ]:
# concatenate all the models
forced_trend_da = {}
for interval in variable_name:
    forced_trend_da[interval] = xr.concat([CanESM5_forced_trend_da[interval],IPSL_forced_trend_da[interval],
                         EC_Earth_forced_trend_da[interval],ACCESS_forced_trend_da[interval],
                         MPI_ESM_forced_trend_da[interval],MIROC6_forced_trend_da[interval]],dim='run')

In [ ]:
forced_trend_da

In [ ]:
forced_trend_da_scale = {}
for interval in variable_name:
    forced_trend_da_scale[interval] = forced_trend_da[interval].assign_coords({'run':np.arange(1,244,1)})*10.0

In [ ]:
forced_trend_da_scale

In [ ]:
forced_trend_da_scale['10yr'].trend.min().values, forced_trend_da_scale['10yr'].trend.max().values

In [ ]:
# calculate the ensemble mean
forced_trend_da_mean = {}
for interval in variable_name:
    forced_trend_da_mean[interval] = forced_trend_da_scale[interval].mean(dim='run')

In [ ]:
forced_trend_da_mean

In [ ]:
# define the function to calculate the noise pattern by subtracting the ensemble mean from the single realization
# then calculate the standard deviation of the noise pattern
def calculate_noise_pattern(single_realization_da, ensemble_mean_da):
    """
    This function is used to calculate the noise pattern by subtracting the ensemble mean from the single realization
    then calculate the standard deviation of the noise pattern
    """
    # calculate the noise pattern
    noise_pattern_da = single_realization_da - ensemble_mean_da
    
    return noise_pattern_da

In [ ]:
Noise_pattern = {}
for interval in variable_name:
    Noise_pattern[interval] = xr.apply_ufunc(calculate_noise_pattern,
        forced_trend_da_scale[interval].trend.chunk({'run':1}), 
        forced_trend_da_mean[interval].trend,
        input_core_dims=[['lat', 'lon'], ['lat', 'lon']],
        output_core_dims=[['lat', 'lon']],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float])

In [ ]:
Noise_pattern

In [ ]:
# define the function to calculate the noise pattern standard deviation
def calculate_noise_pattern_std(noise_da):
    """
    This function is used to calculate the noise pattern standard deviation
    """
    # calculate the noise pattern standard deviation
    noise_std_da = noise_da.std(dim='run')
    
    return noise_std_da

In [ ]:
Noise_pattern['10yr']

### Plotting with the Robinson Projections

In [ ]:
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap


def plot_trend(trend_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 14}
    gl.ylabel_style = {'size': 14}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj

In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [ ]:
MMEM_annual_noise_std['10yr'].max().values

In [ ]:
lat = MMEM_annual_noise_std['10yr'].lat
lon = MMEM_annual_noise_std['10yr'].lon
lat, lon 

titles = ["2013-2022(10yr)",  "1993-2022(30yr)", "1963-2022(60yr)"]
titles_left = ["a.", "b.", "c."]

import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

# Define the GridSpec
fig = plt.figure(figsize=(25, 15))
gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 1], hspace=0.1)
extend='max'
periods = ["10yr", "30yr", "60yr"]
for j, period in enumerate(periods):
    # Define the axes
    ax = fig.add_subplot(gs[j], projection=ccrs.Robinson(180))
    is_left = (j % 3 == 0)
    is_bottom_row = j >= (len(periods)//3)*3 
    
    trend_data = MMEM_annual_noise_std[period]
    trend_with_cyclic, lon_with_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
    
    levels = np.arange(-1.0, 1.1, 0.1)
    contour_obj = plot_trend(trend_with_cyclic, lat, lon_with_cyclic, 
                    levels=levels, extend=extend, cmap='twilight_shifted',
                    # cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors),
                    title=titles[j], ax=ax, show_xticks = is_bottom_row, show_yticks = True)
    ax.text(-0.03, 1.05, titles_left[j], fontsize=22,weight='bold', ha='center', va='center', rotation='horizontal', transform=ax.transAxes)


# Add horizontal colorbars
cbar_ax = fig.add_axes([0.37, 0.04, 0.3, 0.0125])
cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation='horizontal')
cbar.ax.tick_params(labelsize=14)
cbar.set_label('Std. of Annual SAT Noise Trend (°C/decade)', fontsize=16)

plt.tight_layout()
fig.savefig('MMEM_simulated_Noise_trend_Pattern_variations.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
client.close()
scluster.close()